In [1]:
import duckdb

In [2]:
con =duckdb.connect(database='dados_duckdb.db', read_only=False)

In [3]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(10)



,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-30 19:22:13.968230
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-30 19:22:13.968230
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-05-30 19:22:13.968230
3,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163
4,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-05-30 19:22:50.896163
6,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-30 19:22:50.896163
7,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-30 19:22:50.896163
8,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163
9,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163


In [4]:
# como tirar a duplicada dos registro
#vai adicionar um numero pela ordem de injestao 
df = con.execute("""
    SELECT * , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
    FROM bronze_z0019
    WHERE data_ingestao >= '2026-05-30'
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,1
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,2
2,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163,1
3,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163,2
4,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-30 19:22:13.968230,3
5,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-30 19:22:50.896163,1
6,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-30 19:22:50.896163,2
7,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,1
8,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,2
9,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-30 19:22:13.968230,3


In [6]:

df = con.execute("""
                SELECT * FROM (
                 SELECT * , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                FROM bronze_z0019
                WHERE data_ingestao >= '2026-05-30'
                ) WHERE row = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,1
1,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-05-30 19:22:50.896163,1
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-05-30 19:22:50.896163,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-30 19:22:50.896163,1
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-30 19:22:50.896163,1


In [12]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR": "ID"})
df_final = df_final.rename(columns={"MAKTX": "Nome"})
df_final = df_final.rename(columns={"WERKS": "Categoria"})
df_final = df_final.rename(columns={"MAINS": "Fornecedor"})
df_final = df_final.rename(columns={"LABST": "Preco"})
df_final.head(10)

,ID,Nome,Categoria,Fornecedor,Preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10003,PREGO,BT10,100,50
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [16]:
df_final.dtypes

ID            str
Nome          str
Categoria     str
Fornecedor    str
Preco         str
dtype: object

In [18]:
df2 = df_final
df2 = df2.astype(
    {
        'ID': 'int32',
        'Nome': 'string',
        'Categoria': 'string',
        'Fornecedor': 'int32',
        'Preco': 'float32'
    }
)
df2.dtypes
#df2.head(10)

ID              int32
Nome           string
Categoria      string
Fornecedor      int32
Preco         float32
dtype: object

In [19]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
            ID BIGINT,
            Nome TEXT,
            Categoria TEXT,
            Fornecedor BIGINT,
            Preco FLOAT
            )
""")

In [20]:
df2.head(10)

,ID,Nome,Categoria,Fornecedor,Preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,PREGO,BT10,100,50.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [22]:
con.execute("INSERT INTO produtos SELECT * FROM df2 ")

In [23]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,ID,Nome,Categoria,Fornecedor,Preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,PREGO,BT10,100,50.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [24]:
con.close()